Raven Griffin

# Set Up the Computational Environment

**BINF 6210/8210 - Machine Learning for Bioinformatics**

## Learning objectives
By the end of this notebook, you should be able to:

1. distinguish Python, an environment, a package manager, Jupyter, and a kernel;
2. verify that the notebook is using the intended interpreter;
3. inspect and document package versions;
4. use paths that work across operating systems;
5. make a small ML workflow reproducible; and
6. export enough environment information for another researcher to rerun the work.

> Run cells from top to bottom. A check mark means the corresponding setup check passed.
___


## 1. Five layers of a computational setup

- **Operating system:** Windows, macOS, or Linux.
- **Python interpreter:** reads, translates, and executes Python code on a computer.
- **Virtual environment:** An isolated Python workspace containing a particular Python interpreter and its installed packages. It allows each project to use its own packages and package versions without affecting other projects. It exists on a computer simply as a directory containing Python and its installed packages.
- **Package manager:** A tool used to download, install, update, inspect, and remove software libraries. In Python, `pip` is the most common package manager. Pip stands for a recursive acronym: Pip Installs Packages.
- **Jupyter kernel:** computational engine that acts as the backbone process to run the code contained within a Jupyter notebook. When you execute code in a notebook, the frontend browser interface sends that code to the kernel. The kernel runs the code, manages memory, tracks variables, and passes the output back to your screen.  A Python kernel uses a particular Python interpreter and the packages available in that interpreter’s environment.

___

## 2. TO execute: Do the following sequential steps in a terminal inside your project root folder:
- Create a virtual environment

` python -m venv .venv`

You should see a folder named `.venv` in your project root folder.

- Activate the virtual environment.

macOS/Linux: `source .venv/bin/activate`

Windows: `.venv\Scripts\activate.bat`

After the activation, you should see `.venv` in the path.

- Execute:

`python -m pip list`

You should see `pip` exist as a package and its version.

- Install `ipykernel`

`python -m pip install ipykernel`

Executing `python -m pip list` should produce a long list of packages in your environment.

- Install packages that the project will need:

`python -m pip install numpy pandas matplotlib scikit-learn`

- In the notebook, select the interpreter located in your virtual environment.

- Deactivate the environment.

`deactivate`

___

## TO execute (optional): Delete a virtual environment when you need to create a clean environment

Procedure:

   - If the environment is active, deactivate it: `deactivate`
   - Delete the environment folder:
        - macOS/Linux: `rm -rf .venv`
        - Windows Command Prompt: `rmdir /s /q .venv`

___

To insert the symbol ✓ shown in the cell below, you can use several ways:
- Mac: Press `Control + Command + Space`, search for check mark, then select ✓.
- Windows: Press `Windows + .` to open the symbol panel, select Symbols, and look for ✓.
- Copy and paste: ✓

___

A terminal may show one activated environment while Jupyter uses a different kernel. Therefore, verify `sys.executable`; do not rely only on the environment name shown in the terminal or notebook menu.

In [121]:
import os
import platform
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent

print(f"Python version : {sys.version.split()[0]}")
print(f"Interpreter    : {sys.executable}")
print(f"Operating system: {platform.platform()}")
print(f"Working folder : {project_root}")
print("✓ Interpreter information collected")


Python version : 3.12.4
Interpreter    : /opt/anaconda3/bin/python
Operating system: macOS-15.6.1-arm64-i386-64bit
Working folder : /Users/ravengriffin
✓ Interpreter information collected


### Think:

1. Is `sys.executable` located inside the environment you intended to use?
2. How is the working folder different from the folder containing the notebook?
3. Why can `pip install package` affect the wrong Python installation?

**Recommended terminal form:** `python -m pip install package` because it explicitly invokes `pip` through the selected Python interpreter.


## 3. Verify the core scientific Python stack

This course will use NumPy, pandas, Matplotlib, scikit-learn, Jupyter, and later TensorFlow or PyTorch. Deep-learning frameworks are deliberately excluded from this first environment because they are large and hardware-dependent.


In [124]:
from importlib.metadata import PackageNotFoundError, version

core_packages = [
    "numpy", "pandas", "matplotlib", "scikit-learn",
    "jupyterlab", "ipykernel"
]

installed = {}
for package in core_packages:
    try:
        installed[package] = version(package)
    except PackageNotFoundError:
        installed[package] = "NOT INSTALLED"

for package, package_version in installed.items():
    print(f"{package:15s} {package_version}")

missing = [name for name, value in installed.items() if value == "NOT INSTALLED"]
print("\nStatus:", "✓ Core stack available" if not missing else f"Install: {', '.join(missing)}")


numpy           2.5.2
pandas          2.2.3
matplotlib      3.10.7
scikit-learn    1.9.0
jupyterlab      4.3.4
ipykernel       6.29.5

Status: ✓ Core stack available


### If a package is missing

Run this in a **terminal with your environment activated**, not by automatically adding installation commands to a research notebook:

```bash
python -m pip install -r requirements.txt
```

Keeping installation separate makes the notebook easier to audit and avoids modifying an environment unexpectedly. Restart the Jupyter kernel after installing packages.


## 4. Run a scientific-computing smoke test

A successful import is useful, but a small calculation tests that the packages work together.


In [127]:
import numpy as np
import pandas as pd
import sklearn

X_smoke = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
assert X_smoke.shape == (3, 2)
assert np.isfinite(X_smoke).all()

smoke_df = pd.DataFrame(X_smoke, columns=["feature_1", "feature_2"])
display(smoke_df)
print(f"✓ NumPy {np.__version__}, pandas {pd.__version__}, and scikit-learn {sklearn.__version__} work")

,feature_1,feature_2
0,1.0,2.0
1,3.0,4.0
2,5.0,6.0


✓ NumPy 2.5.2, pandas 2.2.3, and scikit-learn 1.9.0 work


## 5. Use portable project paths

A reproducible project should not contain machine-specific paths such as `C:\Users\name\...` or `/Users/name/...`. The following pattern assumes this structure:

```text
project/
├── data/
├── notebooks/
├── results/
├── src/
└── requirements.txt
```

`pathlib` selects the correct path separator for Windows, macOS, or Linux. In a real project, define the project root once and build all paths from it.


In [129]:
data_dir = project_root / "data"
results_dir = project_root / "results"
report_dir = project_root / "report"

print("Data path   :", data_dir)
print("Results path:", results_dir)
print("Report path:", report_dir)
print("✓ Portable paths constructed")

Data path   : /Users/ravengriffin/data
Results path: /Users/ravengriffin/results
Report path: /Users/ravengriffin/report
✓ Portable paths constructed


### Data provenance checkpoint

For every external dataset, record:

- source repository and stable identifier or URL;
- download/access date and dataset version;
- license or data-use restrictions;
- checksum for the raw file when feasible; and
- every transformation from raw data to analysis-ready data.

Sensitive human data should not be committed to Git. Use institutionally approved storage and add data paths to `.gitignore`.


## 6. Make a miniature ML experiment reproducible

Randomness can enter through data simulation, train/test splitting, model initialization, resampling, and GPU operations. A seed controls a pseudorandom stream; it does **not** guarantee identical results across all hardware, library versions, or parallel algorithms.


In [132]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

SEED = 6210
X, y = make_classification(
    n_samples=300, n_features=12, n_informative=5, n_redundant=2,
    class_sep=1.0, random_state=SEED
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
model = LogisticRegression(max_iter=1000, random_state=SEED)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy:.3f}")
print(f"Train/test sizes: {len(y_train)}/{len(y_test)}")
assert 0.0 <= accuracy <= 1.0
print("✓ Reproducible baseline completed")


Test accuracy: 0.840
Train/test sizes: 225/75
✓ Reproducible baseline completed


In [133]:
# Rerun the complete experiment and compare the result.
def run_experiment(seed):
    X2, y2 = make_classification(
        n_samples=300, n_features=12, n_informative=5, n_redundant=2,
        class_sep=1.0, random_state=seed
    )
    Xtr, Xte, ytr, yte = train_test_split(
        X2, y2, test_size=0.25, stratify=y2, random_state=seed
    )
    m = LogisticRegression(max_iter=1000, random_state=seed).fit(Xtr, ytr)
    return accuracy_score(yte, m.predict(Xte))

repeat_accuracy = run_experiment(SEED)
print(f"Original: {accuracy:.6f}; repeated: {repeat_accuracy:.6f}")
assert np.isclose(accuracy, repeat_accuracy)
print("✓ Same code + same seed + same environment produced the same result")


Original: 0.840000; repeated: 0.840000
✓ Same code + same seed + same environment produced the same result


### Experiment (3 minutes)

Change `SEED` in the call below. Does the score change? Explain why a different score is not evidence that the code is incorrect.


In [135]:
run_experiment(seed=32)

0.8933333333333333

In [136]:
run_experiment(seed=42)


0.8533333333333334

##### Different seeds produce different sample of data so the score should naturally change. There is no evidence that the code is incorrect

## 7. Check hardware without making it a requirement

Classical scikit-learn workflows generally run well on a CPU. We will use a GPU only when it materially helps later deep-learning work. The checks below are optional and do not install anything.


In [139]:
print("Logical CPU cores:", os.cpu_count())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    mps = getattr(torch.backends, "mps", None)
    print("Apple Metal (MPS) available:", bool(mps and mps.is_available()))
except ImportError:
    print("PyTorch not installed.")

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
    print("Visible GPUs:", tf.config.list_physical_devices("GPU"))
except ImportError:
    print("TensorFlow not installed.")


Logical CPU cores: 14
PyTorch not installed.
TensorFlow not installed.


## 8. Capture reproducibility metadata

A full `pip freeze` is useful for an exact snapshot, while a short environment report is easier to read in a manuscript supplement or troubleshooting request. `pip freeze` can include transitive and platform-specific packages, so do not treat it as a carefully designed cross-platform dependency file.


In [141]:
import json
from datetime import datetime, timezone

report = {
    "captured_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "executable": sys.executable,
    "platform": platform.platform(),
    "packages": installed,
    "random_seed": SEED,
}
print(json.dumps(report, indent=2))


{
  "captured_utc": "2026-08-27T14:44:39.948918+00:00",
  "python": "3.12.4",
  "executable": "/opt/anaconda3/bin/python",
  "platform": "macOS-15.6.1-arm64-i386-64bit",
  "packages": {
    "numpy": "2.5.2",
    "pandas": "2.2.3",
    "matplotlib": "3.10.7",
    "scikit-learn": "1.9.0",
    "jupyterlab": "4.3.4",
    "ipykernel": "6.29.5"
  },
  "random_seed": 6210
}


In [142]:
# Optional: save a readable report in the reports folder.
report_file = report_dir / "environment_report.json"
report_file.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"✓ Wrote {report_file}")


✓ Wrote /Users/ravengriffin/report/environment_report.json


## 9. What is in your virtual environment?

`python -m pip freeze > requirements.txt`

The resulting requirementx.txt records all packages installed in the environment, including indirect dependencies.

A manually maintained requirements.txt typically lists only the direct packages the project needs.

## 10. Environment health checklist

Before starting an analysis, confirm:

- [ ] the environment is activated;
- [ ] `sys.executable` points to the expected interpreter;
- [ ] the Jupyter kernel matches that interpreter;
- [ ] required packages import successfully;
- [ ] paths are relative to a documented project root;
- [ ] raw data are immutable and provenance is recorded;
- [ ] random seeds are set where appropriate;
- [ ] package versions and operating-system details are captured; and
- [ ] the notebook can run from top to bottom after restarting the kernel.


## Answer these questions:

1. What is the difference between a virtual environment and a Jupyter kernel?
 
 
    Jupyter kernel is a computational engine that acts as the backbone process to run the code contained within a Jupyter notebook.The kernel runs the code, manages memory, tracks variables, and passes the output back to your screen. A Python kernel uses a particular Python interpreter and the packages available in that interpreter’s environment. An isolated Python workspace containing a particular Python interpreter and its installed packages. It allows each project to use its own packages and package versions without affecting other projects. It exists on a computer simply as a directory containing Python and its installed packages.

2. Why is a random seed necessary but sometimes insufficient for exact reproducibility?

   A random seed initializes a pseudorandom number generator so that random operations—such as data splitting, sampling, and model initialization—can produce the same results when the code is rerun.

However, using the same seed may not guarantee exactly identical results because outcomes can also depend on:

    - Package and Python versions
    - Operating system and hardware
    - CPU versus GPU computation
    - Parallel processing and the order of operations
    - Nondeterministic algorithms, particularly some GPU operations
    - Differences in the code, data, or preprocessing

Therefore, a seed is necessary for controlling known sources of randomness, but exact reproducibility also requires documenting the code, data, software environment, hardware, and relevant computational settings.
